# Taxis

In this task I work with the date and time of taxi trips in New York.

Columns I will use:

| Column | Meaning |
|---|---|
| pickup, dropoff | start and end time of the trip |
| distance | distance in miles |
| fare | price of the trip |
| tip | tip in dollars |
| payment | cash or credit card |

## Libraries

- pandas: tables and csv files
- numpy: arrays
- plotly express: plots
- seaborn: only to download the data

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns

## Loading the data

First I download the data, save it as csv, and read it again with pandas.

In [9]:
sns.load_dataset("taxis").to_csv("taxis.csv", index=False)

df = pd.read_csv("taxis.csv")
df.head()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


## DataFrame

In [10]:
df.shape

(6433, 14)

In [12]:
df.info()

<class 'pandas.DataFrame'>
Index: 6341 entries, 0 to 6432
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pickup           6341 non-null   str    
 1   dropoff          6341 non-null   str    
 2   passengers       6341 non-null   int64  
 3   distance         6341 non-null   float64
 4   fare             6341 non-null   float64
 5   tip              6341 non-null   float64
 6   tolls            6341 non-null   float64
 7   total            6341 non-null   float64
 8   color            6341 non-null   str    
 9   payment          6341 non-null   str    
 10  pickup_borough   6341 non-null   str    
 11  dropoff_borough  6341 non-null   str    
dtypes: float64(5), int64(1), str(6)
memory usage: 1.1 MB


In [ ]:
df.isna().sum()

In [11]:
df.drop(columns=["pickup_zone", "dropoff_zone"], inplace=True)
df.dropna(inplace=True)
df.shape

(6341, 12)

In [ ]:
df.groupby("payment")["tip"].mean()

In [ ]:
df.groupby("pickup_borough")["fare"].mean()

## Datetime



In [13]:
df["pickup"] = pd.to_datetime(df["pickup"])
df["dropoff"] = pd.to_datetime(df["dropoff"])
df[["pickup", "dropoff"]].dtypes

pickup     datetime64[us]
dropoff    datetime64[us]
dtype: object

In [14]:
df["hour"] = df["pickup"].dt.hour
df["weekday"] = df["pickup"].dt.day_name()
df["duration_min"] = (df["dropoff"] - df["pickup"]).dt.total_seconds() / 60

df[["pickup", "hour", "weekday", "duration_min"]].head()

,pickup,hour,weekday,duration_min
0,2019-03-23 20:21:09,20,Saturday,6.250000
1,2019-03-04 16:11:55,16,Monday,7.083333
2,2019-03-27 17:53:01,17,Wednesday,7.400000
3,2019-03-10 01:23:59,1,Sunday,25.866667
4,2019-03-30 13:27:42,13,Saturday,9.533333


In [15]:
trips_hour = df.groupby("hour").size().reset_index(name="trips")
trips_hour.head()

,hour,trips
0,0,198
1,1,109
2,2,97
3,3,65
4,4,55


In [16]:
trips_day = df.groupby("weekday").size().reset_index(name="trips")
trips_day

,weekday,trips
0,Friday,1091
1,Monday,700
2,Saturday,1028
3,Sunday,857
4,Thursday,896
5,Tuesday,811
6,Wednesday,958


In [17]:
daily_fare = df.set_index("pickup").resample("D")["fare"].mean().reset_index()
daily_fare.head()

,pickup,fare
0,2019-02-28,5.000000
1,2019-03-01,12.393941
2,2019-03-02,12.072165
3,2019-03-03,12.951437
4,2019-03-04,13.272000


## ndarray

In [18]:
fares = df["fare"].to_numpy()

print(fares.shape)
print(fares.ndim)
print(fares.mean())

(6341,)
1
12.887930925721495


In [ ]:
cheap = fares[fares < 10]
print(len(cheap))

In [ ]:
data = df[["distance", "fare", "tip"]].to_numpy()
print(data.shape)
print(data.mean(axis=0))

## Numeric

In [19]:
df[["distance", "fare", "tip", "duration_min"]].describe()

,distance,fare,tip,duration_min
count,6341.000000,6341.000000,6341.000000,6341.000000
mean,2.997707,12.887931,1.972703,14.349183
std,3.719775,10.722249,2.361897,11.523417
min,0.000000,1.000000,0.000000,0.050000
25%,0.990000,6.500000,0.000000,6.533333
50%,1.650000,9.500000,1.750000,10.933333
75%,3.200000,15.000000,2.820000,18.483333
max,36.700000,150.000000,23.190000,107.666667


In [ ]:
fare = df["fare"]

print("mean:", fare.mean())
print("median:", fare.median())
print("variance:", fare.var())
print("std:", fare.std())
print("skew:", fare.skew())
print("kurtosis:", fare.kurt())

In [ ]:
df[["distance", "fare", "tip", "duration_min"]].corr()

## Plots

In [20]:
fig = px.line(trips_hour, x="hour", y="trips", title="Trips per hour")
fig.show()

In [21]:
fig = px.bar(trips_day, x="weekday", y="trips",
             category_orders={"weekday": ["Monday", "Tuesday", "Wednesday", "Thursday",
                                          "Friday", "Saturday", "Sunday"]},
             title="Trips per weekday")
fig.show()

In [22]:
fig = px.line(daily_fare, x="pickup", y="fare", title="Average fare per day")
fig.show()

In [23]:
fig = px.scatter(df, x="distance", y="fare", color="payment", title="Distance vs fare")
fig.show()

## Conclusion

- The number of trips is different in each hour of the day.
- A longer trip has a higher fare.
- Most of the trips are short.